# 13 · Model 1 user profiles (teacher LLM → text profile → embedding)

Builds one short text profile per user with an LLM, from the user's **training items only**, then turns each profile
into a vector for Model 1's distillation (Severin et al., ECIR 2026). The output is read by Ruchika's
`src/data/profile_file.py` (PR #14).

### Run it in Google Colab
1. **Runtime → Change runtime type → GPU.** An A100 or L4 is best; a T4 also works (the LLM is then loaded in 4-bit).
2. Set `MODE` in the first code cell: `'pilot'` first (200 users), then `'full'` (all 56,017), later `'cold'`.
3. **Runtime → Run all.** Allow Google Drive access when asked.

The full run saves its progress after every chunk. If Colab disconnects, just run all cells again: finished chunks are
skipped.

### What it reads (frozen data on the shared Drive, never changed)
* `sequences.parquet`: `user_idx`, `train_items` (oldest to newest), `n_train`. **Only `train_items` is used;
  `val_item` and `test_item` are never read.**
* `item_text.parquet`: `item_idx`, `item_text`. From each product's `item_text` only the Title, Brand, Category and
  Price parts are kept (Features and Description are dropped to keep prompts short). No review text is used.
* `interactions.parquet`: `user_idx`, `item_idx`, `rating`, `split`. **Only rows with `split == 'train'` are used**, to
  give each training item the user's own star rating.
* `cold_start/cold_sequences.parquet` and `cold_start/cold_items.npy` (cold mode only).

### What it writes (to the shared Drive, never to Git)
| File | Contents |
|---|---|
| `user_profiles.parquet` | `user_idx`, `profile_text`, `teacher_model`, `prompt_version`, `n_train_items_used` |
| `profiles_<encoder>.npz` | `user_idx` (int64), `embeddings` (float32), `source = "llm_profile"`, `train_items_only = True` |
| `profiles_meta.json` | teacher model and revision, exact prompt, encoder and revision, dimension, normalization, date, selection rule |
| `failed_users.json` | every user without a profile, with the reason (no silent skips) |
| `pilot_report.json` | pilot only: failure rate, length, seconds per profile, estimate for all users |
| `review_sample.csv` | pilot only: 20 profiles next to their items, for reading by hand |

In [1]:
# ============================== SETTINGS (edit here) ==============================
MODE = 'pilot'                     # 'pilot' (200 users) -> 'full' (all 56,017) -> 'cold' (cold-start variant)

DATA_DIR = '/content/drive/MyDrive/298A/04_Data/NewDataRunFiles/processed'   # frozen data (read only)
OUT_ROOT = '/content/drive/MyDrive/298A/model1_profiles'                     # results go to <mode>_<prompt version>, e.g. pilot_v2

# Teacher LLM: decide with Abhinita. The paper used Gemma-2-9B; any pinned instruction-tuned model works.
TEACHER_MODEL = 'Qwen/Qwen2.5-7B-Instruct'   # or 'google/gemma-2-9b-it': gated, so first accept its terms on
                                             # huggingface.co and run: from huggingface_hub import login; login()
TEACHER_REVISION = None                      # None = use the current commit and record it; or paste a commit id
MAX_NEW_TOKENS = 160
BATCH_SIZE = 16                              # lowered automatically if the GPU runs out of memory

# Encoders for the vectors. Pilot: MiniLM only. Final runs: ['e5_large', 'minilm'] (your and Abhinita's call).
ENCODERS = ['minilm']
ENCODER_REVISIONS = {}                       # e.g. {'minilm': '<commit id>'}; empty = current commit, recorded

PROMPT_VERSION = 'v2'                        # change when the prompt changes; results go to <mode>_<version>/
MAX_ITEMS = 50                               # use the most recent 50 training items
CHUNK_SIZE = 2000                            # users per saved chunk in full / cold mode
SEED = 42
PILOT_STRATA = [(3, 4, 100), (5, 9, 60), (10, None, 40)]   # (min n_train, max n_train, users) -> 200 users
COST_PER_GPU_HOUR_USD = None                 # optional, for the cost estimate in the pilot report

DRY_RUN = False                              # True = test the plumbing with stand-in models (no GPU, no real output)
# ==================================================================================

In [2]:
import os, sys, json, re, time, math, hashlib, datetime, random
from pathlib import Path
import numpy as np
import pandas as pd

for k in ('MODE', 'DATA_DIR', 'OUT_ROOT', 'TEACHER_MODEL'):            # environment overrides, for local testing
    if os.environ.get(f'PROFILES_{k}'):
        globals()[k] = os.environ[f'PROFILES_{k}']
if os.environ.get('PROFILES_DRY_RUN'):
    DRY_RUN = os.environ['PROFILES_DRY_RUN'] == '1'
if os.environ.get('PROFILES_CHUNK_SIZE'):
    CHUNK_SIZE = int(os.environ['PROFILES_CHUNK_SIZE'])
if os.environ.get('PROFILES_STOP_AFTER_CHUNKS'):                      # test hook: simulate a crash after N chunks
    STOP_AFTER_CHUNKS = int(os.environ['PROFILES_STOP_AFTER_CHUNKS'])
else:
    STOP_AFTER_CHUNKS = None

assert MODE in ('pilot', 'full', 'cold'), MODE
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR = Path(DATA_DIR)
OUT_DIR = Path(OUT_ROOT) / (f'{MODE}_{PROMPT_VERSION}' if not DRY_RUN else f'dry_run_{MODE}_{PROMPT_VERSION}')
CHUNK_DIR = OUT_DIR / 'chunks'
CHUNK_DIR.mkdir(parents=True, exist_ok=True)
assert (DATA_DIR / 'sequences.parquet').exists(), f'sequences.parquet not found in {DATA_DIR}'
random.seed(SEED); np.random.seed(SEED)
print('MODE      :', MODE, '(DRY RUN: stand-in models, output is NOT usable)' if DRY_RUN else '')
print('Data      :', DATA_DIR)
print('Output    :', OUT_DIR)

Mounted at /content/drive
MODE      : pilot 
Data      : /content/drive/MyDrive/298A/04_Data/NewDataRunFiles/processed
Output    : /content/drive/MyDrive/298A/model1_profiles/pilot_v2


## 1. Load the training histories and the product text
Only `user_idx`, `train_items` and `n_train` are loaded, so validation and test items cannot reach a prompt.

In [3]:
if MODE == 'cold':
    seq = pd.read_parquet(DATA_DIR / 'cold_start' / 'cold_sequences.parquet', columns=['user_idx', 'train_items', 'n_train'])
    cold_items = set(np.load(DATA_DIR / 'cold_start' / 'cold_items.npy').tolist())
else:
    seq = pd.read_parquet(DATA_DIR / 'sequences.parquet', columns=['user_idx', 'train_items', 'n_train'])
    cold_items = set()
seq['train_items'] = seq.train_items.apply(lambda a: [int(i) for i in a])
assert seq.user_idx.is_unique and (seq.train_items.str.len() == seq.n_train).all()

items = pd.read_parquet(DATA_DIR / 'item_text.parquet', columns=['item_idx', 'item_text'])
assert items.item_idx.is_unique and items.item_text.str.len().gt(0).all()

FIELDS_KEPT = ('Title', 'Brand', 'Category', 'Price')
_FIELD_SPLIT = re.compile(r' \| (?=(?:Title|Brand|Category|Features|Description|Price|Reviews): )')

def short_item_line(item_text, title_words=30):
    # keep the Title, Brand, Category and Price parts of item_text (metadata only)
    parts = {}
    for part in _FIELD_SPLIT.split(item_text):
        key, _, value = part.partition(': ')
        if key in FIELDS_KEPT and value.strip() and value.strip() != '...':
            value = value.strip()
            if key == 'Title':
                words = value.split()
                value = ' '.join(words[:title_words]) + (' ...' if len(words) > title_words else '')
            parts[key] = value
    return ' | '.join(f'{k}: {parts[k]}' for k in FIELDS_KEPT if k in parts)

inter = pd.read_parquet(DATA_DIR / 'interactions.parquet', columns=['user_idx', 'item_idx', 'rating', 'split'])
assert set(inter.split.unique()) <= {'train', 'val', 'test'}
train_r = inter[inter.split == 'train']                                  # validation and test rows are never used
RATING = dict(zip(zip(train_r.user_idx.astype(int), train_r.item_idx.astype(int)), train_r.rating.astype(float)))
assert len(RATING) == len(train_r), 'expected one training rating per user-item pair'
print(f'Training ratings loaded: {len(RATING):,} (validation and test rows ignored: {len(inter) - len(train_r):,})')
del inter, train_r

ITEM_LINE = dict(zip(items.item_idx, items.item_text.map(short_item_line)))
assert all(ITEM_LINE.values()), 'every item needs at least a title'
print(f'{len(seq):,} users, {len(items):,} items')
print('Example item line:', next(iter(ITEM_LINE.values())))

Training ratings loaded: 389,365 (validation and test rows ignored: 112,034)
56,017 users, 25,903 items
Example item line: Title: SmartLab Toys Go Natural Deluxe Lip Balm Lab with 20 Recipes, Tips and Techniques. STEM Science Lab | Brand: SmartLab | Category: Skin Care > Lip Care > Balms & Moisturizers


## 2. Choose the users
* **pilot**: 200 users drawn with seed 42 by training-history length: 100 with 3 to 4 items, 60 with 5 to 9, 40 with 10 or more.
* **full**: all users. **cold**: all users of the cold-start variant (every user still has at least one training item).

In [4]:
if MODE == 'pilot':
    rng = np.random.default_rng(SEED)
    picks = []
    for lo, hi, n in PILOT_STRATA:
        pool = seq[(seq.n_train >= lo) & ((seq.n_train <= hi) if hi is not None else True)].user_idx.sort_values().to_numpy()
        assert len(pool) >= n, (lo, hi, len(pool))
        picks.append(np.sort(rng.choice(pool, size=n, replace=False)))
    users = seq.set_index('user_idx').loc[np.concatenate(picks)].reset_index()
    SELECTION_RULE = (f'pilot: seed {SEED}, numpy default_rng, without replacement within strata of n_train: '
                      + ', '.join(f"{n} users with n_train {lo}-{hi if hi is not None else 'max'}" for lo, hi, n in PILOT_STRATA))
else:
    users = seq.copy()
    SELECTION_RULE = f'{MODE}: all users in ' + ('cold_start/cold_sequences.parquet' if MODE == 'cold' else 'sequences.parquet')

users['items_used'] = users.train_items.apply(lambda l: l[-MAX_ITEMS:])
users['n_train_items_used'] = users.items_used.str.len()
users['ratings_used'] = [[RATING.get((int(u), i)) for i in l] for u, l in zip(users.user_idx, users.items_used)]
n_missing = sum(r is None for l in users.ratings_used for r in l)
assert n_missing == 0, f'{n_missing} used items have no training rating'
print('PASS every item in a prompt has its own training rating (split == train)')
no_items = users[users.n_train_items_used == 0]
users = users[users.n_train_items_used > 0].reset_index(drop=True)
if cold_items:
    assert not any(i in cold_items for l in users.items_used for i in l), 'a cold item would reach a profile'
    print('PASS no cold item is used in any cold-variant profile')
print(SELECTION_RULE)
print(f'{len(users):,} users to profile; {len(no_items)} users have no training items')
print(users.n_train_items_used.describe().round(1).to_dict())

PASS every item in a prompt has its own training rating (split == train)
pilot: seed 42, numpy default_rng, without replacement within strata of n_train: 100 users with n_train 3-4, 60 users with n_train 5-9, 40 users with n_train 10-max
200 users to profile; 0 users have no training items
{'count': 200.0, 'mean': 7.9, 'std': 9.2, 'min': 3.0, '25%': 3.0, '50%': 4.5, '75%': 7.2, 'max': 50.0}


## 3. The prompt (version v2)
The exact text is saved in `profiles_meta.json`. Products are listed oldest to newest, each with the user's own training
star rating. No user id, review, validation or test item is ever included.

v2 (after reading the v1 pilot): defines 3 stars as neutral and asks for dislikes by name; describes prices only where
they are shown (no "all products ..."); no claims that the shopper avoids something in their list; needs only when the
titles state them (the v1 example list was copied into profiles); no vague praise; "natural"/"organic" only when most
titles say so.

In [5]:
PROMPT_TEMPLATE = '''You are helping a skin-care shop understand what one of its shoppers likes.

Below are the products this shopper rated, from oldest to newest. Each line is the product's catalog information (with a price band only when the catalog has one), followed by the shopper's own star rating (1 to 5 stars).

{item_lines}

Write a short profile of this shopper's skin-care interests in 2 to 4 sentences (at most 100 words).

How to read the ratings:
- 4 or 5 stars: the shopper liked the product.
- 3 stars: neutral. Do not describe it as liked or disliked.
- 1 or 2 stars: the shopper disliked the product. If there are any, name the disliked product types or brands.

What to describe:
- The product types, categories and brands of the liked products.
- Price level, only from the prices actually shown. Many products have no price, so never say "all" or "always" about prices unless every product shows one. If few prices are shown, leave price out.
- Needs or benefits only when the product titles state them.

Rules:
- Use only what the product list shows. Do not invent products, brands, prices, needs or benefits.
- Never say the shopper is not interested in, avoids or does not care about something that appears in the list.
- Do not use vague praise such as "high-quality" or "effective". Say "natural" or "organic" only if most of the listed titles say so.
- Describe products and interests only. Do not guess or state any health condition, age, gender, ethnicity or other personal identity.
- Refer to the person as "this shopper". Do not use lists, headings or any ids.
- Reply with the profile text only.'''
PROMPT_SHA = hashlib.sha256(PROMPT_TEMPLATE.encode()).hexdigest()

def build_prompt(item_ids, ratings):
    return PROMPT_TEMPLATE.format(item_lines='\n'.join(
        f'{k}. {ITEM_LINE[i]} | Rating: {r:g}/5' for k, (i, r) in enumerate(zip(item_ids, ratings), 1)))

users['prompt'] = [build_prompt(i, r) for i, r in zip(users.items_used, users.ratings_used)]
assert not users.prompt.str.contains(r'(?:Features|Description|Reviews): ').any(), 'only Title/Brand/Category/Price allowed'
print('PASS prompts contain only training items, with Title, Brand, Category and Price (no reviews, no val/test items)')
ex = users.sort_values('n_train_items_used').iloc[0]
print(f'Example prompt ({ex.n_train_items_used} items):\n' + '-' * 60 + '\n' + ex.prompt)

PASS prompts contain only training items, with Title, Brand, Category and Price (no reviews, no val/test items)
Example prompt (3 items):
------------------------------------------------------------
You are helping a skin-care shop understand what one of its shoppers likes.

Below are the products this shopper rated, from oldest to newest. Each line is the product's catalog information (with a price band only when the catalog has one), followed by the shopper's own star rating (1 to 5 stars).

1. Title: Medline Remedy Phytoplex Nourishing Skin Cream, Skin Moisturizer, Paraben Free Body Lotion, 16 Fl Oz, 2 Pack | Brand: Remedy Dermatology Series | Category: Skin Care > Body > Moisturizers > Creams | Price: $10-25 | Rating: 5/5
2. Title: Neutrogena Cleansing Fragrance Free Makeup Remover Face Wipes, Cleansing Facial Towelettes for Waterproof Makeup, Alcohol-Free, Unscented, 100% Plant-Based Fibers, Twin Pack, 2 x 25 ct | Brand: Neutrogena | Category: Skin Care > Face > Cleansers > Cloths

## 4. Load the teacher LLM
The model's exact commit is resolved once and recorded, so the run can be reproduced. Generation is greedy
(temperature 0), at most 160 new tokens.

In [6]:
import torch
torch.manual_seed(SEED)
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
print('Device:', GPU_NAME)

def resolve_revision(repo_id, pinned):
    if pinned or DRY_RUN or Path(repo_id).exists():
        return pinned or ('local' if Path(repo_id).exists() else 'dry-run')
    from huggingface_hub import HfApi
    return HfApi().model_info(repo_id).sha

if DRY_RUN:
    TEACHER_ID, TEACHER_REV, LOAD_NOTE = 'DRY-RUN stand-in (no LLM)', 'dry-run', 'none'
    def generate_batch(prompts):
        out = []
        for p in prompts:
            brands = re.findall(r'Brand: ([^|\n]+)', p)
            cats = re.findall(r'Category: ([^|\n]+)', p)
            out.append(f"This shopper mostly buys {cats[-1].strip() if cats else 'skin care'} products"
                       + (f", often from {brands[-1].strip()}." if brands else '.'))
        return out
else:
    import transformers
    from transformers import AutoTokenizer, AutoModelForCausalLM
    _v = tuple(int(x) for x in re.findall(r'\d+', transformers.__version__)[:2])
    DTYPE_KW = 'dtype' if _v >= (4, 56) else 'torch_dtype'               # the argument was renamed in 4.56
    TEACHER_ID = TEACHER_MODEL
    TEACHER_REV = resolve_revision(TEACHER_MODEL, TEACHER_REVISION)
    tok = AutoTokenizer.from_pretrained(TEACHER_MODEL, revision=None if TEACHER_REV == 'local' else TEACHER_REV)
    tok.padding_side = 'left'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kwargs = {}
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
    if torch.cuda.is_available() and vram_gb < 20:
        import subprocess, importlib.util                               # T4 (16 GB): 4-bit weights
        if importlib.util.find_spec('bitsandbytes') is None:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes'], check=True)
        from transformers import BitsAndBytesConfig
        kwargs['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                                           bnb_4bit_compute_dtype=torch.float16)
        LOAD_NOTE = f'4-bit NF4 (GPU has {vram_gb:.0f} GB)'
    elif torch.cuda.is_available():
        kwargs[DTYPE_KW] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        LOAD_NOTE = str(kwargs[DTYPE_KW])
    else:
        kwargs[DTYPE_KW] = torch.float32
        LOAD_NOTE = 'float32 on CPU (very slow; use a GPU)'
    if 'gemma-2' in TEACHER_MODEL.lower():
        kwargs['attn_implementation'] = 'eager'                          # recommended for Gemma 2
    model = AutoModelForCausalLM.from_pretrained(TEACHER_MODEL, revision=None if TEACHER_REV == 'local' else TEACHER_REV,
                                                 device_map='auto' if torch.cuda.is_available() else None, **kwargs)
    model.eval()

    def generate_batch(prompts):
        texts = [tok.apply_chat_template([{'role': 'user', 'content': p}], tokenize=False, add_generation_prompt=True)
                 for p in prompts]
        enc = tok(texts, return_tensors='pt', padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        return tok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)

print('Teacher:', TEACHER_ID, '| revision:', TEACHER_REV, '| loaded as:', LOAD_NOTE)

Device: NVIDIA L4


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Teacher: Qwen/Qwen2.5-7B-Instruct | revision: a09a35458c702b33eeacc393d103063234e8bc28 | loaded as: torch.bfloat16


## 5. Generate the profiles
Users are processed in chunks; each finished chunk is saved to `chunks/` immediately, so a crash loses at most one
chunk. A user whose generation fails or comes back empty is recorded with the reason, never dropped silently.

In [7]:
def clean(text):
    text = re.sub(r'\s+', ' ', str(text)).strip()
    text = re.sub(r'^(profile|shopper profile)\s*:\s*', '', text, flags=re.I).strip().strip('"').strip()
    return text

def run_users(df):
    # generate for one chunk; returns one row per user with status 'ok' or 'failed'
    df = df.assign(_len=df.prompt.str.len()).sort_values('_len', ascending=False)   # similar lengths per batch
    rows, bs, i = [], BATCH_SIZE, 0
    while i < len(df):
        batch = df.iloc[i:i + bs]
        t0 = time.time()
        try:
            outs = generate_batch(batch.prompt.tolist())
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs == 1:
                outs = [None] * len(batch)
                err = 'out of GPU memory'
            else:
                bs = max(1, bs // 2); print(f'  out of memory -> batch size {bs}'); continue
        except Exception as e:                                           # record, never skip silently
            outs, err = [None] * len(batch), f'{type(e).__name__}: {e}'[:300]
        secs = (time.time() - t0) / len(batch)
        for (_, u), o in zip(batch.iterrows(), outs):
            text = clean(o) if o is not None else ''
            status = 'ok' if text else 'failed'
            reason = '' if text else (err if o is None else 'empty output')
            rows.append(dict(user_idx=int(u.user_idx), profile_text=text, status=status, reason=reason,
                             n_train_items_used=int(u.n_train_items_used), seconds=secs,
                             prompt_sha=PROMPT_SHA, teacher=f'{TEACHER_ID}@{TEACHER_REV}'))
        i += len(batch)
    return pd.DataFrame(rows)

chunks = [users] if MODE == 'pilot' else [users.iloc[s:s + CHUNK_SIZE] for s in range(0, len(users), CHUNK_SIZE)]
t_start, done_now = time.time(), 0
for c, part in enumerate(chunks):
    path = CHUNK_DIR / f'chunk_{c:05d}.parquet'
    if path.exists():
        prev = pd.read_parquet(path)
        same_users = sorted(prev.user_idx) == sorted(part.user_idx)
        same_setup = ('prompt_sha' in prev and prev.prompt_sha.eq(PROMPT_SHA).all()
                      and prev.teacher.eq(f'{TEACHER_ID}@{TEACHER_REV}').all())
        if same_users and same_setup:
            continue                                                     # finished in an earlier session
        print(f'chunk {c + 1}: saved result was made with a different prompt or model -> regenerating')
    if STOP_AFTER_CHUNKS is not None and done_now >= STOP_AFTER_CHUNKS:
        raise SystemExit(f'test hook: simulated crash after {done_now} chunk(s)')
    res = run_users(part)
    res.to_parquet(path.with_suffix('.tmp'), index=False)
    os.replace(path.with_suffix('.tmp'), path)                           # atomic: no half-written chunk
    done_now += 1
    el = time.time() - t_start
    print(f'chunk {c + 1}/{len(chunks)}: {len(res)} users, {(res.status == "failed").sum()} failed, {el / 60:.1f} min so far')

res = pd.concat([pd.read_parquet(CHUNK_DIR / f'chunk_{c:05d}.parquet') for c in range(len(chunks))], ignore_index=True)
assert res.user_idx.is_unique and set(res.user_idx) == set(users.user_idx), 'every selected user must have a result'
print(f'{len(res):,} users: {(res.status == "ok").sum():,} profiles, {(res.status == "failed").sum():,} failed')

  out of memory -> batch size 8
  out of memory -> batch size 4
chunk 1/1: 200 users, 0 failed, 6.1 min so far
200 users: 200 profiles, 0 failed


## 6. Save the profiles and the failed users

In [8]:
ok = res[res.status == 'ok'].sort_values('user_idx')
profiles = pd.DataFrame({
    'user_idx': ok.user_idx.astype('int64'),
    'profile_text': ok.profile_text.astype(str),
    'teacher_model': f'{TEACHER_ID}@{TEACHER_REV}',
    'prompt_version': PROMPT_VERSION,
    'n_train_items_used': ok.n_train_items_used.astype('int64'),
}).reset_index(drop=True)
profiles.to_parquet(OUT_DIR / 'user_profiles.parquet', index=False)

failed = [dict(user_idx=int(r.user_idx), reason=r.reason) for r in res[res.status == 'failed'].itertuples()]
failed += [dict(user_idx=int(u), reason='no training items') for u in no_items.user_idx]
(OUT_DIR / 'failed_users.json').write_text(json.dumps(failed, indent=1))
assert len(profiles) + len(failed) == len(users) + len(no_items), 'every user is either profiled or listed as failed'
print(f'Saved user_profiles.parquet ({len(profiles):,} rows) and failed_users.json ({len(failed)} users)')

Saved user_profiles.parquet (200 rows) and failed_users.json (0 users)


## 7. Checks to read by hand
Automatic flags only help the reading; they do not reject anything. **Read the 20 profiles in `review_sample.csv`** and
check: (1) nothing that is not derivable from the listed products, (2) no sensitive inference (health, age, gender,
ethnicity, identity), (3) sensible length.

In [9]:
words = profiles.profile_text.str.split().str.len()
SENSITIVE = re.compile(r"\b(she|he|her|his|woman|women|man|men|female|male|girl|boy|teen\w*|age[ds]?|old|young|"
                       r"years|mother|mom|pregnan\w*|baby|wife|husband|asian|black|white|latin\w*|african|"
                       r"diabet\w*|eczema|psoriasis|rosacea|cancer|disease|condition|allerg\w*|acne)\b", re.I)
per_user = users.set_index('user_idx')
item_blob = {u: ' '.join(ITEM_LINE[i] for i in l).lower() for u, l in zip(users.user_idx, users.items_used)}

def sensitive_terms(uid, text):
    # terms from the list that are NOT simply copied from the user's own product text (e.g. brand "Jack Black")
    found = {m.lower() for m in SENSITIVE.findall(text)}
    return ', '.join(sorted(t for t in found if not re.search(rf'\b{re.escape(t)}\b', item_blob[uid])))

LOW_MENTION = re.compile(r"dislik|low(er)?[- ]rat|poor|[12][- ]star|[12]/5|did not like|didn't like|less favou?r|negative", re.I)
has_low = per_user.ratings_used.apply(lambda rr: any(r <= 2 for r in rr))
flags = pd.DataFrame({
    'user_idx': profiles.user_idx,
    'words': words,
    'over_120_words': words > 120,
    'possible_sensitive_terms': [sensitive_terms(u, t) for u, t in zip(profiles.user_idx, profiles.profile_text)],
    'mentions_digits_like_ids': profiles.profile_text.str.contains(r'\b\d{4,}\b'),
    'says_all_about_price': profiles.profile_text.str.contains(r"\b(?:all|every|always)\b[^.]{0,90}(?:\$|price)", case=False),
    'claims_absence': profiles.profile_text.str.contains(r"not (?:a )?(?:major )?concern|no (?:apparent )?interest|not interested|avoids?\b|does not care", case=False),
    'vague_praise': profiles.profile_text.str.contains(r"high[- ]quality|\beffective\b", case=False),
    'dislikes_not_mentioned': [bool(has_low[u]) and not LOW_MENTION.search(t) for u, t in zip(profiles.user_idx, profiles.profile_text)],
})
print(f'words per profile: mean {words.mean():.1f}, median {words.median():.0f}, max {words.max()}')
FLAG_COLS = ['over_120_words', 'mentions_digits_like_ids', 'says_all_about_price', 'claims_absence', 'vague_praise',
             'dislikes_not_mentioned']
print('profiles flagged:', {c: int(flags[c].sum()) for c in FLAG_COLS},
      '| possible_sensitive_terms:', int((flags.possible_sensitive_terms != '').sum()))
flags.to_csv(OUT_DIR / 'profile_flags.csv', index=False)

sample = profiles.sample(min(20, len(profiles)), random_state=SEED)
sample = sample.merge(users[['user_idx', 'items_used']], on='user_idx')
sample = sample.merge(users[['user_idx', 'ratings_used']], on='user_idx')
sample['items'] = [' \n'.join(f'{ITEM_LINE[i]} | Rating: {r:g}/5' for i, r in zip(l, rr))
                   for l, rr in zip(sample.items_used, sample.ratings_used)]
flag_text = flags.set_index('user_idx')[FLAG_COLS].apply(lambda r: ', '.join(c for c in FLAG_COLS if r[c]), axis=1)
sample = sample.merge(flags[['user_idx', 'possible_sensitive_terms']], on='user_idx')
sample['auto_flags'] = sample.user_idx.map(flag_text)
sample[['user_idx', 'n_train_items_used', 'profile_text', 'possible_sensitive_terms', 'auto_flags', 'items']].to_csv(
    OUT_DIR / 'review_sample.csv', index=False)
for r in sample.head(5).itertuples():
    print(f'\n--- user {r.user_idx} ({r.n_train_items_used} items) ---\n{r.profile_text}')
print(f'\nAll {len(sample)} review profiles with their items: {OUT_DIR / "review_sample.csv"}')

words per profile: mean 42.7, median 42, max 65
profiles flagged: {'over_120_words': 0, 'mentions_digits_like_ids': 0, 'says_all_about_price': 13, 'claims_absence': 17, 'vague_praise': 31, 'dislikes_not_mentioned': 32} | possible_sensitive_terms: 0

--- user 27470 (5 items) ---
This shopper prefers natural and hypoallergenic skin care products for both body and face, showing a particular liking for body washes and toners. They have rated several products highly, including those from Dove, Aveeno, Bioré, Pond's, and Dickinson's, all of which are priced under $50.

--- user 5449 (3 items) ---
This shopper prefers natural and organic products, particularly self-tanners and lip balms. They seem to value effectiveness and quality, giving high ratings to Fake Bake self-tanners and Artnaturals lip balms. The Mario Badescu Drying Lotion received a neutral rating.

--- user 9227 (3 items) ---
This shopper prefers natural and affordable skincare products, showing a liking for face scrubs, cleans

## 8. Pilot report (pilot mode)
Seconds per profile are measured per stratum and weighted by how many users of each length the full data has, so the
estimate is not skewed by the pilot's over-sampling of long histories.

In [10]:
REPORT = None
if MODE == 'pilot':
    full_seq = pd.read_parquet(DATA_DIR / 'sequences.parquet', columns=['user_idx', 'n_train'])
    timed = res.merge(users[['user_idx', 'n_train']], on='user_idx')
    est_secs, strata = 0.0, []
    for lo, hi, n in PILOT_STRATA:
        in_s = lambda s: (s >= lo) & ((s <= hi) if hi is not None else True)
        sec = timed[in_s(timed.n_train)].seconds.mean()
        n_full = int(in_s(full_seq.n_train).sum())
        est_secs += sec * n_full
        strata.append(dict(n_train=f"{lo}-{hi if hi is not None else 'max'}", pilot_users=n, users_in_full_data=n_full,
                           seconds_per_profile=round(float(sec), 3)))
    hours = est_secs / 3600
    REPORT = {
        'users': int(len(res)), 'profiles': int(len(profiles)), 'failed': int(len(failed)),
        'failure_rate': round(len(failed) / max(1, len(res)), 4),
        'mean_words': round(float(words.mean()), 1), 'median_words': float(words.median()),
        'seconds_per_profile_pilot_mean': round(float(res.seconds.mean()), 3),
        'by_history_length': strata,
        'estimated_hours_all_56017_users': round(hours, 2),
        'estimated_cost_usd': None if COST_PER_GPU_HOUR_USD is None else round(hours * COST_PER_GPU_HOUR_USD, 2),
        'gpu': GPU_NAME, 'teacher': f'{TEACHER_ID}@{TEACHER_REV}', 'loaded_as': LOAD_NOTE, 'batch_size': BATCH_SIZE,
        'flags': {**{c: int(flags[c].sum()) for c in FLAG_COLS},
                  'possible_sensitive_terms': int((flags.possible_sensitive_terms != '').sum())},
        'prompt_version': PROMPT_VERSION,
        'dry_run': DRY_RUN,
    }
    (OUT_DIR / 'pilot_report.json').write_text(json.dumps(REPORT, indent=2))
    print(json.dumps(REPORT, indent=2))

{
  "users": 200,
  "profiles": 200,
  "failed": 0,
  "failure_rate": 0.0,
  "mean_words": 42.7,
  "median_words": 42.0,
  "seconds_per_profile_pilot_mean": 1.772,
  "by_history_length": [
    {
      "n_train": "3-4",
      "pilot_users": 100,
      "users_in_full_data": 32133,
      "seconds_per_profile": 1.41
    },
    {
      "n_train": "5-9",
      "pilot_users": 60,
      "users_in_full_data": 17276,
      "seconds_per_profile": 1.707
    },
    {
      "n_train": "10-max",
      "pilot_users": 40,
      "users_in_full_data": 6608,
      "seconds_per_profile": 2.775
    }
  ],
  "estimated_hours_all_56017_users": 25.87,
  "estimated_cost_usd": null,
  "gpu": "NVIDIA L4",
  "teacher": "Qwen/Qwen2.5-7B-Instruct@a09a35458c702b33eeacc393d103063234e8bc28",
  "loaded_as": "torch.bfloat16",
  "batch_size": 16,
  "flags": {
    "over_120_words": 0,
    "mentions_digits_like_ids": 0,
    "says_all_about_price": 13,
    "claims_absence": 17,
    "vague_praise": 31,
    "dislikes_not_menti

## 9. Embed the profiles and write `profiles_<encoder>.npz`
Vectors are L2-normalized. The encoder's exact commit is resolved and recorded. Ruchika's code reduces the vectors to
the model's size, so any encoder with at least 128 dimensions works.

In [11]:
if not DRY_RUN and 'model' in globals():
    import gc
    del model; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

ENCODER_SPECS = {   # name: (model id, text prefix)
    'minilm': ('sentence-transformers/all-MiniLM-L6-v2', ''),
    'e5_large': ('intfloat/multilingual-e5-large', 'query: '),
}
for k in ('minilm', 'e5_large'):
    if os.environ.get(f'PROFILES_ENCODER_{k.upper()}'):                 # local testing
        ENCODER_SPECS[k] = (os.environ[f'PROFILES_ENCODER_{k.upper()}'], ENCODER_SPECS[k][1])

ENCODER_META = {}
for name in ENCODERS:
    enc_id, prefix = ENCODER_SPECS[name]
    texts = [prefix + t for t in profiles.profile_text]
    if DRY_RUN:
        rev = 'dry-run'
        emb = np.stack([np.random.default_rng(int(hashlib.sha256(t.encode()).hexdigest()[:8], 16)).standard_normal(384)
                        for t in texts]).astype(np.float32)
        emb /= np.linalg.norm(emb, axis=1, keepdims=True)
    else:
        from sentence_transformers import SentenceTransformer
        rev = resolve_revision(enc_id, ENCODER_REVISIONS.get(name))
        st = SentenceTransformer(enc_id, revision=None if rev == 'local' else rev,
                                 device='cuda' if torch.cuda.is_available() else 'cpu')
        emb = st.encode(texts, batch_size=128, normalize_embeddings=True, show_progress_bar=True,
                        convert_to_numpy=True).astype(np.float32)
        del st
    uid = profiles.user_idx.to_numpy(dtype=np.int64)
    path = OUT_DIR / f'profiles_{name}.npz'
    np.savez(path, user_idx=uid, embeddings=emb, source=np.array('synthetic' if DRY_RUN else 'llm_profile'),
             train_items_only=np.array(True))
    ENCODER_META[name] = dict(model=enc_id, revision=rev, prefix=prefix, dim=int(emb.shape[1]), l2_normalized=True,
                              file=path.name, sha256=hashlib.sha256(path.read_bytes()).hexdigest())
    print(f'{path.name}: {emb.shape}, encoder {enc_id}@{rev}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

profiles_minilm.npz: (200, 384), encoder sentence-transformers/all-MiniLM-L6-v2@1110a243fdf4706b3f48f1d95db1a4f5529b4d41


## 10. Check the files (Ruchika's checks) and write `profiles_meta.json`

In [12]:
for name in ENCODERS:
    z = np.load(OUT_DIR / f'profiles_{name}.npz', allow_pickle=False)
    uid, emb = z['user_idx'], z['embeddings']
    assert z['train_items_only'].dtype == bool and z['train_items_only'].ndim == 0 and bool(z['train_items_only'])
    assert z['source'].ndim == 0 and str(z['source']) == ('synthetic' if DRY_RUN else 'llm_profile')
    assert uid.ndim == 1 and len(np.unique(uid)) == len(uid) and uid.min() >= 0 and uid.max() <= 56016
    assert emb.ndim == 2 and emb.shape[0] == len(uid) and np.isfinite(emb).all()
    assert uid.dtype == np.int64 and emb.dtype == np.float32
    assert (uid == profiles.user_idx.to_numpy()).all(), 'npz rows must match user_profiles.parquet'
    print('OK', name, emb.shape)

meta = {
    'mode': MODE, 'dry_run': DRY_RUN,
    'created_utc': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'teacher_model': TEACHER_ID, 'teacher_revision': TEACHER_REV, 'teacher_loaded_as': LOAD_NOTE, 'gpu': GPU_NAME,
    'generation': {'do_sample': False, 'temperature': 0, 'max_new_tokens': MAX_NEW_TOKENS, 'seed': SEED,
                   'batch_size': BATCH_SIZE},
    'prompt_version': PROMPT_VERSION, 'prompt_template': PROMPT_TEMPLATE, 'prompt_sha256': PROMPT_SHA,
    'item_text_fields_kept': list(FIELDS_KEPT), 'max_items_per_user': MAX_ITEMS, 'items_order': 'oldest to newest',
    'inputs': {'sequences': 'cold_start/cold_sequences.parquet' if MODE == 'cold' else 'sequences.parquet',
               'columns_read': ['user_idx', 'train_items', 'n_train'], 'item_text': 'item_text.parquet (item_text only)',
               'ratings': "interactions.parquet: user_idx, item_idx, rating where split == 'train' only"},
    'selection_rule': SELECTION_RULE, 'seed': SEED,
    'n_users_selected': int(len(users) + len(no_items)), 'n_profiles': int(len(profiles)), 'n_failed': int(len(failed)),
    'encoders': ENCODER_META,
    'pilot_report': REPORT,
}
(OUT_DIR / 'profiles_meta.json').write_text(json.dumps(meta, indent=2))
print('Files in', OUT_DIR, ':', sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))

OK minilm (200, 384)
Files in /content/drive/MyDrive/298A/model1_profiles/pilot_v2 : ['failed_users.json', 'pilot_report.json', 'profile_flags.csv', 'profiles_meta.json', 'profiles_minilm.npz', 'review_sample.csv', 'user_profiles.parquet']


## 11. Make sure everything has reached Google Drive
Colab writes to Drive in the background. This waits until every file is synced (and disconnects Drive, so run it last).

In [13]:
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.flush_and_unmount()
    print('All files are saved to Google Drive:', OUT_DIR)
else:
    print('Not in Colab: files are in', OUT_DIR)

All files are saved to Google Drive: /content/drive/MyDrive/298A/model1_profiles/pilot_v2
